In [1]:
pip install --upgrade azureml-sdk azureml-contrib-dataset cython pycocotools

Requirement already up-to-date: azureml-sdk in /home/cody/miniconda3/envs/aml/lib/python3.7/site-packages (1.0.72)
Requirement already up-to-date: azureml-contrib-dataset in /home/cody/miniconda3/envs/aml/lib/python3.7/site-packages (1.0.72.2)
  Using cached https://files.pythonhosted.org/packages/d8/58/2deb24de3c10cc4c0f09639b46f4f4b50059f0fdc785128a57dd9fdce026/Cython-0.29.14-cp37-cp37m-manylinux1_x86_64.whl
  Using cached https://files.pythonhosted.org/packages/96/84/9a07b1095fd8555ba3f3d519517c8743c2554a245f9476e5e39869f948d2/pycocotools-2.0.0.tar.gz
    ERROR: Command errored out with exit status 1:
     command: /home/cody/miniconda3/envs/aml/bin/python -c 'import sys, setuptools, tokenize; sys.argv[0] = '"'"'/tmp/pip-install-8uf_75pw/pycocotools/setup.py'"'"'; __file__='"'"'/tmp/pip-install-8uf_75pw/pycocotools/setup.py'"'"';f=getattr(tokenize, '"'"'open'"'"', open)(__file__);code=f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"

In [ ]:
import torch
import utils
from azureml.core import Workspace, Dataset, Datastore
from azureml.contrib.dataset import Dataset


subscription_id = 'f375b912-331c-4fc5-8e9f-2d7205e3e036'
resource_group = 'LabelingDemoRG'
workspace_name = 'labeling_demo_1'

workspace = Workspace(subscription_id, resource_group, workspace_name)

labeled_dataset = Dataset.get_by_name(workspace, name='Domestic Animals Detection-2019-10-26 00:35:21')
labeled_dataset.label['type'] = 'ObjectDetection'
labeled_dataset.image['column'] = 'image_url'
pytorch_dataset = labeled_dataset._to_torchvision()

In [ ]:
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
# from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pytorch_obj_detection.engine import train_one_epoch, evaluate


def get_model_instance_segmentation(num_classes):
    # load an instance segmentation model pre-trained pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def foo(): 
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    num_classes = 4

    indices = torch.randperm(len(pytorch_dataset)).tolist()
    dataset = torch.utils.data.Subset(pytorch_dataset, indices[:126])
    dataset_test = torch.utils.data.Subset(pytorch_dataset, indices[-20:])
    data_loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=utils.collate_fn)
    length = len(data_loader)
    data_loader_test = torch.utils.data.DataLoader(dataset_test, batch_size=2, shuffle=False, num_workers=0, collate_fn=utils.collate_fn)
    model = get_model_instance_segmentation(num_classes)
    model.to(device)


    # construct an optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    # and a learning rate scheduler
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    num_epochs = 10
    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
        # update the learning rate
        lr_scheduler.step()
        # evaluate on the test dataset
        evaluate(model, data_loader_test, device=device)
    pass


if __name__ == '__main__':
    foo()